# 2 · The model explained

What sets the pH in the cell, and how the reactions, the precipitation of ZHS and transport interact. Equations: `docs/model.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model
from znmno2_model.eqchem import Equilibria

## Speciation: most of the zinc is complexed

In 2 M ZnSO₄ most Zn²⁺ is bound in ZnSO₄⁰ and Zn(SO₄)₂²⁻. Titrating the electrolyte with acid or base (changing the proton total H_T, with the sulfate set by electroneutrality) shows the species against pH, as in Herrmann et al. (2023), Fig. S1. The dashed line is ZHS saturation.

In [ ]:
eq = Equilibria()
zn, mn = 2.0, 0.5
H = np.linspace(-0.06, 0.4, 300)
T = np.stack([H, np.full_like(H, zn), np.full_like(H, mn), zn + mn + H / 2], axis=1)
x = eq.solve(T)
m = eq.species(x)
pH = -x[:, 0]
for name in ("Zn+2", "ZnSO4", "Zn(SO4)2-2", "ZnOH+", "Zn(OH)2"):
    plt.plot(pH, m[:, eq.names.index(name)] / zn, label=name)
w = np.exp((4 * np.log(m[:, 1]) + np.log(m[:, 3]) - 28.4 * np.log(10)) / 6) / m[:, 0]
o = np.argsort(w)
plt.axvline(np.interp(1.0, w[o], pH[o]), ls="--", c="gray", label="ZHS saturation")
plt.yscale("log"); plt.ylim(1e-6, 1.2); plt.xlabel("pH"); plt.ylabel("fraction of Zn"); plt.legend(fontsize=8);

## Inside the cell during a discharge

`keep_states=True` keeps the full state at every output row; `Model.profiles` turns a state into profiles across the cell (anode at x = 0, then the probe region, the separator and the cathode).

In [ ]:
p = Params(steps="cc I=100 Vmin=1.0", write_interval=600.0)
r = run(p, keep_states=True)
model = Model(p)
picks = np.linspace(0, len(r.states) - 1, 6).astype(int)
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for i in picks:
    t, x = r.states[i]
    pr = model.profiles(x)
    lab = f"{r.rows[i][3]:.0f} mAh/g"
    ax[0, 0].plot(pr["x_um"], pr["pH"], label=lab)
    ax[0, 1].plot(pr["x_um"], pr["Mn_M"], label=lab)
    ax[1, 0].plot(pr["x_um"], pr["vf_ZHS"], label=lab)
    ax[1, 1].plot(pr["x_um"], pr["theta"], label=lab)
for a, yl in zip(ax.flat, ("pH", "Mn total [M]", "ZHS volume fraction", "insertion fraction θ")):
    a.set(xlabel="x [µm]", ylabel=yl)
ax[0, 0].legend(fontsize=8)
for a in ax.flat[2:]:
    a.set_xlim(model.mesh.x[model.mesh.c0] * 1e4 - 20, None)
fig.tight_layout();

Mn²⁺ released by R2 dissolution diffuses out of the cathode into the separator and the probe region. That matters on charge, when R2 deposition needs it back.

## Why ZHS pins the pH

Once ZHS is present and precipitation is fast, the cathode pH stays where the solution is saturated with ZHS: the protons consumed by MnO₂ dissolution are replaced by ZHS formation (4Zn²⁺ + SO₄²⁻ + 6H₂O → ZHS + 6H⁺). Comparing the cathode pH with the ZHS saturation pH of the local solution:

In [ ]:
q = r.column("mAhg")
sat = []
for t, x in r.states:
    free = model.free(x)
    c = model.cath
    w = model.zhs_w(free)                       # c_H,eq / c_H in every cathode cell
    sat.append(np.mean(-np.log10(free[c, 0] * w)))
plt.plot(q, r.column("pH_cath"), label="cathode pH")
plt.plot(q, sat, "--", label="ZHS saturation pH of the local solution")
plt.xlabel("capacity [mAh/g]"); plt.ylabel("pH"); plt.legend();

## Conservation

Zn, Mn, S and H are conserved exactly (to round-off). Zn changes only by what the anode supplies: the charge passed / 2F.

In [ ]:
inv0, inv1 = model.inventory(r.states[0][1]), model.inventory(r.states[-1][1])
q_C = r.column("mAhg")[-1] * p.mass * 3.6
for k in ("Mn", "S", "H"):
    print(f"{k}: change {inv1[k] - inv0[k]:+.2e} mol (of {inv0[k] if k != 'H' else inv0['S']:.2e})")
print(f"Zn: change {inv1['Zn'] - inv0['Zn']:.6e} mol; charge / 2F = {q_C / (2 * p.F):.6e} mol")